# Movie Recommendation System — Phase 2: Models & Evaluation
**Models:** Global Average Baseline · User-Based Collaborative Filtering (KNN)  
**Library:** Surprise (`pip install scikit-surprise`)  
**Course:** CSIT 360 — Spring 2026

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from surprise import Dataset, Reader, NormalPredictor, KNNBasic, KNNWithMeans, SVD
from surprise.model_selection import cross_validate, train_test_split, GridSearchCV
from surprise import accuracy

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
os.makedirs('../results/figures', exist_ok=True)

print('Imports OK.')

## 2. Load Cleaned Data into Surprise

In [ ]:
df = pd.read_csv('../data/processed/ratings_clean.csv')
print(f'Loaded {len(df):,} ratings.')

# Tell Surprise the rating scale
reader = Reader(rating_scale=(1, 5))

# Load into Surprise Dataset format
data = Dataset.load_from_df(df[['userId', 'movieId', 'rating']], reader)

# 80/20 train-test split
trainset, testset = train_test_split(data, test_size=0.20, random_state=42)

print(f'Train size: {trainset.n_ratings:,} ratings')
print(f'Test  size: {len(testset):,} ratings')

## 3. Model 1 — Global Average Baseline

**Intuition:** The simplest possible model. Every prediction is just the overall average rating across all users and movies. It ignores who the user is or what movie they're rating. It sets the floor — any real recommender should beat this.

In [ ]:
from surprise import NormalPredictor
# NormalPredictor predicts a random value based on the distribution of the training set.
# For a true "global mean" baseline, we use the built-in BaselineOnly with no regularization,
# or simply compute it manually. Let's use Surprise's BaselineOnly with bsl_options:
from surprise import BaselineOnly

# Global average: bias toward global mean (mu), with very light regularization
bsl_options = {'method': 'sgd', 'n_epochs': 20}
model_baseline = BaselineOnly(bsl_options=bsl_options, verbose=False)

model_baseline.fit(trainset)
preds_baseline = model_baseline.test(testset)

rmse_baseline = accuracy.rmse(preds_baseline, verbose=False)
mae_baseline  = accuracy.mae(preds_baseline,  verbose=False)

print(f'Global Average Baseline')
print(f'  RMSE : {rmse_baseline:.4f}')
print(f'  MAE  : {mae_baseline:.4f}')

## 4. Model 2 — User-Based Collaborative Filtering (KNN)

**Intuition:** Find users whose past ratings are similar to the target user (neighbors). Predict ratings based on what those similar users rated. Uses **cosine similarity** to measure how alike two users are. The key parameter is `k` (how many neighbors to use).

In [ ]:
# Hyperparameter tuning: try different k values via cross-validation
print('Tuning k for User-Based CF (this may take a minute)...')

param_grid = {'k': [10, 20, 40], 'sim_options': [
    {'name': 'cosine', 'user_based': True},
    {'name': 'pearson', 'user_based': True}
]}

gs = GridSearchCV(KNNWithMeans, param_grid, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
gs.fit(data)

print(f'Best RMSE : {gs.best_score["rmse"]:.4f}')
print(f'Best params: {gs.best_params["rmse"]}')

In [ ]:
# Train best model on full train set and evaluate on test set
best_k   = gs.best_params['rmse']['k']
best_sim = gs.best_params['rmse']['sim_options']

model_ubcf = KNNWithMeans(k=best_k, sim_options=best_sim, verbose=False)
model_ubcf.fit(trainset)
preds_ubcf = model_ubcf.test(testset)

rmse_ubcf = accuracy.rmse(preds_ubcf, verbose=False)
mae_ubcf  = accuracy.mae(preds_ubcf,  verbose=False)

print(f'User-Based CF  (k={best_k}, sim={best_sim["name"]})')
print(f'  RMSE : {rmse_ubcf:.4f}')
print(f'  MAE  : {mae_ubcf:.4f}')

## 5. Comparison Table

In [ ]:
results = pd.DataFrame({
    'Model':       ['Global Average Baseline', f'User-Based CF (k={best_k})'],
    'RMSE':        [round(rmse_baseline, 4), round(rmse_ubcf, 4)],
    'MAE':         [round(mae_baseline,  4), round(mae_ubcf,  4)],
    'Description': [
        'Predicts global mean ± user/item bias',
        f'k nearest users, {best_sim["name"]} similarity'
    ]
})

# Save metrics
results.to_csv('../results/metrics.csv', index=False)
print('Saved → results/metrics.csv')
results

## 6. Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

models     = results['Model']
colors     = ['#5b8db8', '#e07b54']

for ax, metric in zip(axes, ['RMSE', 'MAE']):
    bars = ax.bar(models, results[metric], color=colors, edgecolor='white', width=0.5)
    for bar, val in zip(bars, results[metric]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f'{val:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold'
        )
    ax.set_title(f'Model Comparison — {metric}', fontsize=13, fontweight='bold')
    ax.set_ylabel(metric, fontsize=12)
    ax.set_ylim(0, max(results[metric]) * 1.2)
    ax.tick_params(axis='x', labelsize=9)

plt.suptitle('CiaoDVD Recommendation System: Model Evaluation', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/model_comparison.png', bbox_inches='tight')
plt.show()
print('Saved → results/figures/model_comparison.png')

## 7. Discussion

*(Write your own 3-5 sentence discussion here after running the notebook.)*

**Template:**
> The User-Based CF model achieved an RMSE of **X.XXXX** compared to the Global Average Baseline of **X.XXXX**, representing an improvement of roughly X%. This makes intuitive sense because CF leverages actual user similarity patterns rather than just the population mean. The choice of k and similarity function (cosine vs. Pearson) had a measurable impact on performance, as shown by the cross-validation grid search. Despite the dataset's high sparsity (~XX%), the CF model was still able to find meaningful neighbors for most users.

## ✅ Phase 2 Complete
**Artifacts produced:**
- `results/metrics.csv` — RMSE and MAE for both models
- `results/figures/model_comparison.png` — bar chart

Next → `03_evaluation.ipynb` (optional: Precision@K / Recall@K for bonus points)